In [9]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# WavTokenizer Audio Tokenization Example

This notebook demonstrates the WavTokenizer audio tokenizer - a SOTA discrete codec model achieving 40 tokens per second for audio language modeling.

**Key Properties:**
- Input sample rate: 24 kHz
- Output sample rate: 24 kHz
- Codebook size: 4096 tokens
- Token rate: 40 Hz (40 tokens per second)
- Architecture: Encoder + Decoder with single codebook quantization
- Supports speech, audio, and music


## Setup and Imports


In [10]:
import sys
import torch
import numpy as np
from datasets import load_dataset
from IPython.display import Audio, display
import matplotlib.pyplot as plt

# Add our tokenizer module to path
sys.path.append('../src')

# Import our tokenizer registry
from audio_tokenizers import get_tokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


PyTorch version: 2.6.0a0+df5bbc09d1.nv24.11
CUDA available: True
Using device: cuda


## Load ESB Diagnostic AMI Dataset


In [11]:
# Load the AMI clean split
print("Loading ESB diagnostic dataset - AMI clean split...")
dataset = load_dataset('esb/diagnostic-dataset', 'ami', split='clean')

print(f"Dataset loaded with {len(dataset)} samples")
print(f"Sample keys: {dataset[0].keys()}")


Loading ESB diagnostic dataset - AMI clean split...
Dataset loaded with 770 samples
Sample keys: dict_keys(['audio', 'ortho_transcript', 'norm_transcript', 'id', 'dataset'])


## Find Long Audio Samples


In [12]:
# Calculate duration for each sample and find the longest ones
durations = []
for i, sample in enumerate(dataset):
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    duration = len(audio_array) / sr
    durations.append((i, duration, sample['norm_transcript']))

# Sort by duration and get top 5 longest
durations.sort(key=lambda x: x[1], reverse=True)
long_samples = durations[:5]

print("Top 5 longest audio samples:")
for idx, (i, dur, transcript) in enumerate(long_samples):
    print(f"{idx+1}. Sample {i}: {dur:.2f} seconds")
    print(f"   Transcript: {transcript[:100]}..." if len(transcript) > 100 else f"   Transcript: {transcript}")
    print()


Top 5 longest audio samples:
1. Sample 85: 11.77 seconds
   Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c ...

2. Sample 1: 11.70 seconds
   Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would lik...

3. Sample 100: 11.69 seconds
   Transcript: that f the fruit and vegetable theme is the is the current trend and and therefore um we need to go ...

4. Sample 31: 11.44 seconds
   Transcript: and also we talked about um a location function where maybe you could press a button on the t v and ...

5. Sample 210: 11.33 seconds
   Transcript: the the problem is if you have to go across the building and it adds some overhead every time you wa...



## Initialize WavTokenizer


In [13]:
# Initialize the tokenizer
print("Loading WavTokenizer tokenizer...")
tokenizer = get_tokenizer('wavtokenizer', device=device)

print(f"Tokenizer loaded: {tokenizer}")
print(f"  Input sample rate: {tokenizer.sample_rate} Hz")
print(f"  Output sample rate: {tokenizer.output_sample_rate} Hz")
print(f"  Codebook size: {tokenizer.codebook_size:,}")
print(f"  Downsample rate: {tokenizer.downsample_rate}x")
print(f"  Token rate: {tokenizer.tokens_per_second} Hz (40 tokens per second)")


Loading WavTokenizer tokenizer...


Tokenizer loaded: WavTokenizerWrapper(checkpoint='None', device='cuda', sample_rate=24000)
  Input sample rate: 24000 Hz
  Output sample rate: 24000 Hz
  Codebook size: 4,096
  Downsample rate: 600x
  Token rate: 40 Hz (40 tokens per second)


## Tokenize and Reconstruct Audio Samples


In [14]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "128"

# Process the top 3 longest samples
results = []

for idx in range(min(3, len(long_samples))):
    sample_idx, duration, transcript = long_samples[idx]
    sample = dataset[sample_idx]
    
    print(f"\n{'='*60}")
    print(f"Processing Sample {idx+1} (index {sample_idx})")
    print(f"Duration: {duration:.2f} seconds")
    print(f"Transcript: {transcript[:150]}..." if len(transcript) > 150 else f"Transcript: {transcript}")
    
    # Get audio data
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    
    # Convert to tensor and ensure correct shape (1, T) for WavTokenizer
    audio_tensor = torch.from_numpy(audio_array).float()
    if audio_tensor.dim() == 1:
        audio_tensor = audio_tensor.unsqueeze(0)  # (T,) -> (1, T)
    
    # Tokenize
    print("\nTokenizing...")
    tokens, encode_info = tokenizer.encode(audio_tensor, sr=sr)
    
    # Show token information
    print(f"Token shape: {tokens.shape}")
    print(f"Number of tokens: {tokens.numel()}")
    print(f"Tokens per second: {tokens.numel() / duration:.1f}")
    
    # Show first 20 discrete token values
    token_values = tokens.squeeze().cpu().numpy()
    print(f"\nFirst 20 discrete token IDs:")
    print(token_values[:20])
    print(f"Token ID range: [{token_values.min()}, {token_values.max()}]")
    print(f"Unique tokens used: {len(np.unique(token_values))}")
    
    # Decode back to audio
    print("\nDecoding tokens back to audio...")
    reconstructed, decode_info = tokenizer.decode(tokens)
    
    print(f"Reconstructed shape: {reconstructed.shape}")
    print(f"Output sample rate: {decode_info['output_sample_rate']} Hz")
    
    # Handle shape: (1, 1, T) or (1, T) or (T,)
    rec = reconstructed
    if rec.dim() == 3:
        rec = rec[0, 0] if rec.shape[1] == 1 else rec[0]
    elif rec.dim() == 2:
        rec = rec[0]
    rec_np = rec.detach().cpu().numpy() if torch.is_tensor(rec) else rec
    
    # Store results
    results.append({
        'original': audio_array,
        'original_sr': sr,
        'reconstructed': rec_np,
        'reconstructed_sr': decode_info['output_sample_rate'],
        'tokens': token_values,
        'transcript': transcript,
        'duration': duration
    })
    
print(f"\n{'='*60}")
print("Processing complete!")



Processing Sample 1 (index 85)
Duration: 11.77 seconds
Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c i can see like some kind of a thing where like you...

Tokenizing...
Token shape: torch.Size([1, 471])
Number of tokens: 471
Tokens per second: 40.0

First 20 discrete token IDs:
[1106 2038  805  210 3540 3389 2578 3564 1737  839 1571 2259 2638 4091
 1455 1803 2629 3771  753   54]
Token ID range: [0, 4091]
Unique tokens used: 389

Decoding tokens back to audio...
Reconstructed shape: torch.Size([1, 282600])
Output sample rate: 24000 Hz

Processing Sample 2 (index 1)
Duration: 11.70 seconds
Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would like to see in a convenient practical

Tokenizing...
Token shape: torch.Size([1, 468])
Number of tokens: 468
Tokens per second: 40.0

First 20 discrete token IDs:
[1106 2569 2287 3437 2648  697  870 4053 3195 3830 2627 1034  573

## Play Original vs Reconstructed Audio


In [15]:
# Display audio players for comparison
for idx, result in enumerate(results):
    print(f"\n{'='*60}")
    print(f"Sample {idx+1} - Duration: {result['duration']:.2f}s")
    print(f"Transcript: {result['transcript'][:200]}..." if len(result['transcript']) > 200 else f"Transcript: {result['transcript']}")
    print(f"\nTokens used: {len(result['tokens'])} ({len(result['tokens'])/result['duration']:.1f} tokens/sec)")
    
    print(f"\nOriginal Audio ({result['original_sr']} Hz):")
    display(Audio(result['original'], rate=result['original_sr']))
    
    print(f"\nReconstructed Audio ({result['reconstructed_sr']} Hz):")
    display(Audio(result['reconstructed'], rate=result['reconstructed_sr']))
    
    # Compression ratio
    original_size = len(result['original']) * 2  # 16-bit audio
    token_size = len(result['tokens']) * 2  # 12-bit tokens (log2(4096) = 12)
    compression_ratio = original_size / token_size
    print(f"\nCompression ratio: {compression_ratio:.1f}x")



Sample 1 - Duration: 11.77s
Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c i can see like some kind of a thing where like you sort of have like the number come up on the t v l...

Tokens used: 471 (40.0 tokens/sec)

Original Audio (16000 Hz):



Reconstructed Audio (24000 Hz):



Compression ratio: 399.8x

Sample 2 - Duration: 11.70s
Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would like to see in a convenient practical

Tokens used: 468 (40.0 tokens/sec)

Original Audio (16000 Hz):



Reconstructed Audio (24000 Hz):



Compression ratio: 400.0x

Sample 3 - Duration: 11.69s
Transcript: that f the fruit and vegetable theme is the is the current trend and and therefore um we need to go for that if we want

Tokens used: 468 (40.0 tokens/sec)

Original Audio (16000 Hz):



Reconstructed Audio (24000 Hz):



Compression ratio: 399.7x


## Summary Statistics


In [16]:
# Calculate summary statistics
print("Summary of Tokenization Results:")
print("="*60)

total_duration = sum(r['duration'] for r in results)
total_tokens = sum(len(r['tokens']) for r in results)

print(f"Total audio processed: {total_duration:.2f} seconds")
print(f"Total tokens generated: {total_tokens:,}")
print(f"Average tokens per second: {total_tokens/total_duration:.1f}")

# Calculate average compression ratio
avg_compression = np.mean([
    (len(r['original']) * 2) / (len(r['tokens']) * 2) 
    for r in results
])
print(f"Average compression ratio: {avg_compression:.1f}x")

print(f"\nWavTokenizer Properties:")
print(f"  Bitrate: {40 * 12 / 1000:.2f} kbps (approx)")
print(f"  Token rate: 40 Hz")
print(f"  Bits per token: 12 (log2(4096))")
print(f"  Codebook size: 4,096")
print(f"  Input: 24 kHz mono (resampled from 16 kHz)")
print(f"  Output: 24 kHz mono")
print(f"  Model: large-600 (trained on 80,000 hours)")


Summary of Tokenization Results:
Total audio processed: 35.16 seconds
Total tokens generated: 1,407
Average tokens per second: 40.0
Average compression ratio: 399.8x

WavTokenizer Properties:
  Bitrate: 0.48 kbps (approx)
  Token rate: 40 Hz
  Bits per token: 12 (log2(4096))
  Codebook size: 4,096
  Input: 24 kHz mono (resampled from 16 kHz)
  Output: 24 kHz mono
  Model: large-600 (trained on 80,000 hours)


## Summary

WavTokenizer successfully:
- Encodes audio to discrete tokens at 40 Hz (40 tokens per second)
- Uses a single codebook with 4,096 tokens
- Provides efficient compression for audio representation
- Supports speech, audio, and music
- Can decode tokens back to 24 kHz audio

The discrete tokens can be used for:
- Audio compression and storage
- Input to language models for audio generation
- Audio editing and manipulation tasks
- Building blocks for audio synthesis pipelines

**Key Advantages:**
- **40 tokens/second**: Balanced rate for quality and efficiency
- **Single codebook**: Simple architecture, easy to use with standard language models
- **Universal support**: Works well with speech, audio, and music
- **High quality**: Trained on 80,000 hours of diverse audio data
